# TwiBench — full pipeline (Google Colab or Kaggle)

**Google Colab:** *Runtime → Change runtime type → **T4 GPU** → Save*, then *Runtime → Run all*.
At the end, results are saved to your Google Drive in `MyDrive/TwiBench/` (Colab asks once for Drive access).
Keep the browser tab open while it runs.

**Kaggle:** *Session options → Accelerator: GPU*, *Internet: On*, then *Run All*.
Results appear in the *Output* panel.

`twibench_results.zip` (a few MB: metrics, predictions, data stats, training log) is what you
unzip into the project folder on your PC. Total time is about 3–5 hours, most of it fine-tuning.
Use `MAX_STEPS = 200` in the next cell for a quick test run first (~30–40 min).

In [ ]:
RUN_BASELINES = True     # zero-shot Whisper-small and MMS (aka adapter)
RUN_FINETUNE  = True     # fine-tune Whisper-small, then evaluate it
MAX_STEPS     = 4000     # use 200 for a quick test run
SAVE_MODEL_ZIP = True    # ~1 GB zip of the fine-tuned model (needed later for the demo)

import os, torch
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/TwiBench"
os.makedirs(OUT_DIR, exist_ok=True)
assert torch.cuda.is_available(), "No GPU: switch the runtime/accelerator to a GPU first"
print("GPU:", torch.cuda.get_device_name(0), "| results will be saved to", OUT_DIR)

## 0. Project code (generated from `src/`)

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
%%writefile src/normalize.py
"""Step 4: Twi text normalisation, applied identically to references and
model outputs before scoring.

Two levels:
    normalize(text)                    -> standard scoring text
    normalize(text, fold_vowels=True)  -> also folds ɛ->e, ɔ->o, to measure
                                          how much error is only the special letters
"""
import re
import unicodedata

# Apostrophes / quote variants that appear in Twi elisions (n'ani, w'akyi).
_APOSTROPHES = "’‘ʼ`´"

# Common look-alike substitutions for the Twi open vowels.
_VOWEL_VARIANTS = {
    "Ɛ": "ɛ",  # Ɛ -> ɛ
    "Ɔ": "ɔ",  # Ɔ -> ɔ
    "ε": "ɛ",  # Greek epsilon ε -> ɛ
}

_FOLD = str.maketrans({"ɛ": "e", "ɔ": "o"})


def normalize(text: str, fold_vowels: bool = False) -> str:
    if not text:
        return ""
    # Decompose, so accents/tone marks become separate combining characters...
    text = unicodedata.normalize("NFD", text)
    for k, v in _VOWEL_VARIANTS.items():
        text = text.replace(k, v)
    text = text.lower()
    # ...then drop them: Twi orthography does not mark tone, and the data
    # uses accents inconsistently (e.g. "Ńkɔmmɔ").
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = unicodedata.normalize("NFC", text)

    for a in _APOSTROPHES:
        text = text.replace(a, "'")
    # Keep apostrophes inside words (n'ani), drop all other punctuation.
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)
    text = text.replace("_", " ")

    if fold_vowels:
        text = text.translate(_FOLD)
    return re.sub(r"\s+", " ", text).strip()


In [ ]:
%%writefile src/prepare_data.py
"""Steps 2-3: download the Ashesi Twi data, build a speaker- and
sentence-disjoint train/dev/test split, and export dev/test audio locally.

Source: Ashesi Financial Inclusion Speech Dataset (CC-BY-4.0), via the
Hugging Face mirrors below. Original file names encode the metadata, e.g.
    AsantiTwiFm18-MHdyb4ij-Tmp002-YvH2jV.wav
    <dialect><gender><age>-<speaker_id>-<prompt_id>-<clip_id>

Split rule: a speaker and a prompt (sentence) are each hashed into
train/dev/test; a clip is kept only if both land in the same split, so no
test speaker and no test sentence is ever seen in training. Hashing makes
the split deterministic and lets us process one shard at a time.

To save disk space each shard (~450 MB) is downloaded, processed and
deleted. Kept locally (16 kHz FLAC): all dev/test audio plus a fixed ~36%
sample of train audio for CPU fine-tuning.

Usage:
    python src/prepare_data.py
Outputs:
    data/processed/metadata.csv        every clip, parsed fields + split
    data/processed/{train,dev,test}.csv      manifests with local audio paths
    data/processed/audio/{train,dev,test}/   16 kHz FLAC files
    results/data_stats.csv             clips / hours / speakers / prompts
"""
import hashlib
import io
import re
import shutil
import time
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfApi, hf_hub_download

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "data" / "processed"
TMP = ROOT / "data" / "raw" / "tmp"
SR = 16_000

# Fraction of speakers AND of prompts assigned to each evaluation split.
DEV_FRAC = 0.10
TEST_FRAC = 0.15
SALT = "twibench-v1"  # change only if you deliberately want a new split
# Share of train clips whose audio is kept locally (~12k clips) for CPU
# fine-tuning; the GPU notebook reads all train audio from Hugging Face instead.
TRAIN_EXPORT_FRAC = 0.36
MIN_FREE_GB = 1.5  # never fill the disk: stop (resumably) below this
# Shards used per dialect (sorted order: test shard first, then train shards).
# Limited to save internet data: currently Asante only (3 shards).
# Set to None to use every shard of both dialects.
MAX_SHARDS = {"asante": 3, "akuapem": 0}

SOURCES = {
    "asante": "ghanaopenai/twi_multispeaker_audio_transcribed",
    "akuapem": "ghanaopenai/akuapem_multispeaker_audio_transcribed",
}

NAME_RE = re.compile(
    r"^(?P<lang>AsantiTwi|AkuapemTwi)(?P<gender>Fm|Ma|F|M)(?P<age>\d+)"
    r"-(?P<speaker>[^-]+)-(?P<prompt>Tmp\d+)-(?P<clip>[^.]+)\."
)


def unit_hash(key: str) -> float:
    """Deterministic pseudo-random number in [0, 1] for a key."""
    return int(hashlib.sha1(f"{SALT}:{key}".encode()).hexdigest()[:8], 16) / 0xFFFFFFFF


def bucket(key: str) -> str:
    """Deterministically map a speaker or prompt ID to train/dev/test."""
    h = unit_hash(key)
    if h < TEST_FRAC:
        return "test"
    if h < TEST_FRAC + DEV_FRAC:
        return "dev"
    return "train"


def download(repo: str, filename: str, retries: int = 10) -> Path:
    """Download one file, resuming after network drops."""
    for attempt in range(1, retries + 1):
        try:
            return Path(hf_hub_download(repo, filename, repo_type="dataset", local_dir=TMP))
        except Exception as e:
            wait = min(60, 5 * attempt)
            print(f"  download failed ({type(e).__name__}), retry {attempt}/{retries} in {wait}s", flush=True)
            time.sleep(wait)
    raise RuntimeError(f"Could not download {repo}/{filename}")


def parse(df: pd.DataFrame, dialect: str) -> pd.DataFrame:
    parsed = df["file"].str.extract(NAME_RE)
    bad = parsed["speaker"].isna()
    if bad.any():
        print(f"  WARNING: {bad.sum()} unparsed names dropped, e.g. {df.loc[bad, 'file'].head(2).tolist()}")
    df = pd.concat([df, parsed.drop(columns="lang")], axis=1)[~bad].copy()
    df["gender"] = df["gender"].str[0].map({"F": "female", "M": "male"})
    df["age"] = df["age"].astype(int)
    df["speaker"] = dialect + ":" + df["speaker"]
    s, p = df["speaker"].map(bucket), df["prompt"].map(bucket)
    df["split"] = np.where(s == p, s, "unused")
    return df


def process_shard(repo: str, file: str, dialect: str) -> pd.DataFrame:
    local = download(repo, file)
    with open(local, "rb") as fh:
        df = read_shard(pq.ParquetFile(fh), repo, file, dialect)
    local.unlink()  # free disk space (file must be closed first on Windows)
    return df


def read_shard(pf: pq.ParquetFile, repo: str, file: str, dialect: str) -> pd.DataFrame:
    meta = pf.read(columns=["audio_path.path", "transcription"])
    df = pd.DataFrame({
        "file": [r["path"] for r in meta.column("audio_path").to_pylist()],
        "text": meta.column("transcription").to_pylist(),
    })
    df["dialect"], df["hf_repo"], df["hf_file"], df["hf_row"] = dialect, repo, file, np.arange(len(df))
    df = parse(df, dialect)
    df["duration"] = np.nan
    df["audio"] = ""

    # Export dev/test audio and a fixed sample of train audio (for CPU
    # fine-tuning), row group by row group to keep memory low.
    train_sample = (df["split"] == "train") & (df["file"].map(lambda f: unit_hash("clip:" + f)) < TRAIN_EXPORT_FRAC)
    keep = set(df.index[df["split"].isin(["dev", "test"]) | train_sample])
    offset = 0
    for rg in range(pf.metadata.num_row_groups):
        table = pf.read_row_group(rg, columns=["audio_path"])
        n = table.num_rows
        audio = table.column("audio_path").to_pylist()
        for i in range(offset, offset + n):
            if i in df.index:
                df.at[i, "duration"] = round(sf.info(io.BytesIO(audio[i - offset]["bytes"])).duration, 3)
        wanted = [i for i in range(offset, offset + n) if i in keep]
        if wanted:
            for i in wanted:
                y, sr = sf.read(io.BytesIO(audio[i - offset]["bytes"]), dtype="float32")
                if y.ndim > 1:
                    y = y.mean(axis=1)
                if sr != SR:
                    y = librosa.resample(y, orig_sr=sr, target_sr=SR)
                split = df.at[i, "split"]
                rel = Path("data/processed/audio") / split / (Path(df.at[i, "file"]).stem + ".flac")
                (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
                sf.write(ROOT / rel, y, SR)
                df.at[i, "audio"] = rel.as_posix()
                df.at[i, "duration"] = round(len(y) / SR, 3)
        offset += n
    return df


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    (ROOT / "results").mkdir(exist_ok=True)
    TMP.mkdir(parents=True, exist_ok=True)

    # Resume support: shards already processed are cached as small CSVs.
    done_dir = OUT / "shards"
    done_dir.mkdir(exist_ok=True)

    frames = []
    for dialect, repo in SOURCES.items():
        files = sorted(f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet"))
        files = files[:MAX_SHARDS[dialect]] if MAX_SHARDS else files
        for file in files:
            cache = done_dir / f"{dialect}_{Path(file).stem}.csv"
            if cache.exists():
                print("cached ", dialect, file, flush=True)
                frames.append(pd.read_csv(cache))
                continue
            free_gb = shutil.disk_usage(ROOT).free / 1e9
            if free_gb < MIN_FREE_GB:
                raise SystemExit(f"Stopping: only {free_gb:.1f} GB free on disk (need {MIN_FREE_GB} GB). "
                                 "Free some space and re-run; finished shards are skipped.")
            print(f"process {dialect} {file}  ({free_gb:.1f} GB free)", flush=True)
            df = process_shard(repo, file, dialect)
            df.to_csv(cache, index=False, encoding="utf-8")
            frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    dups = df.duplicated(["dialect", "file"]).sum()
    if dups:
        print(f"Dropping {dups} duplicate files")
        df = df.drop_duplicates(["dialect", "file"])

    df.to_csv(OUT / "metadata.csv", index=False, encoding="utf-8")
    cols = ["audio", "text", "dialect", "gender", "age", "speaker", "prompt", "duration", "file"]
    for split in ["dev", "test"]:
        df[df["split"] == split][cols].to_csv(OUT / f"{split}.csv", index=False, encoding="utf-8")
    exported_train = df[(df["split"] == "train") & df["audio"].fillna("").ne("")]
    exported_train[cols].to_csv(OUT / "train.csv", index=False, encoding="utf-8")

    st = (df.groupby(["split", "dialect"])
            .agg(clips=("file", "size"), speakers=("speaker", "nunique"),
                 prompts=("prompt", "nunique"), hours=("duration", lambda d: round(d.sum() / 3600, 2)),
                 female_share=("gender", lambda g: round((g == "female").mean(), 2)))
            .reset_index())
    st.to_csv(ROOT / "results" / "data_stats.csv", index=False)
    print(st.to_string(index=False))
    print("\nTotal clips:", len(df), "| speakers:", df["speaker"].nunique(), "| prompts:", df["prompt"].nunique())

    # Leakage check: no test/dev speaker or prompt may appear in train.
    tr = df[df["split"] == "train"]
    for split in ["dev", "test"]:
        ev = df[df["split"] == split]
        assert not set(ev["speaker"]) & set(tr["speaker"]), f"speaker leak in {split}"
        assert not set(ev["prompt"]) & set(tr["prompt"]), f"prompt leak in {split}"
    print("Leakage check passed: dev/test speakers and prompts are unseen in train.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/prepare_ood.py
"""Out-of-domain test set: scripture-style read Twi (d3vnerd/twi_dataset, MIT).

Used only for testing, to check whether models generalise beyond the
financial domain of the training data.

Usage:
    python src/prepare_ood.py
Outputs:
    data/processed/ood_test.csv
    data/processed/audio/ood_test/*.flac
"""
import io
from pathlib import Path

import librosa
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfApi, hf_hub_download

ROOT = Path(__file__).resolve().parents[1]
REPO = "d3vnerd/twi_dataset"
SR = 16_000
# CPU budget: evaluate on a fixed random sample (the full list is kept in ood_test_full.csv).
OOD_CLIPS = 150


def sample(df: pd.DataFrame) -> pd.DataFrame:
    return df.sample(n=min(OOD_CLIPS, len(df)), random_state=42).sort_index()


def main():
    files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset")
             if f.endswith(".parquet") and "test" in Path(f).name]
    out_dir = ROOT / "data" / "processed" / "audio" / "ood_test"
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for f in files:
        local = Path(hf_hub_download(REPO, f, repo_type="dataset", local_dir=ROOT / "data" / "raw" / "tmp_ood"))
        with open(local, "rb") as fh:
            table = pq.ParquetFile(fh).read()
        audio_col = "audio" if "audio" in table.column_names else table.column_names[0]
        for i, (a, text) in enumerate(zip(table.column(audio_col).to_pylist(),
                                          table.column("transcription").to_pylist())):
            y, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
            if y.ndim > 1:
                y = y.mean(axis=1)
            if sr != SR:
                y = librosa.resample(y, orig_sr=sr, target_sr=SR)
            rel = Path("data/processed/audio/ood_test") / f"{Path(f).stem}_{i:05d}.flac"
            sf.write(ROOT / rel, y, SR)
            rows.append({"audio": rel.as_posix(), "text": text, "duration": round(len(y) / SR, 3)})
        local.unlink()

    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data" / "processed" / "ood_test_full.csv", index=False, encoding="utf-8")
    sub = sample(df)
    sub.to_csv(ROOT / "data" / "processed" / "ood_test.csv", index=False, encoding="utf-8")
    print(f"{len(df)} clips ({df['duration'].sum() / 3600:.2f} h); "
          f"evaluation sample {len(sub)} clips ({sub['duration'].sum() / 3600:.2f} h)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/evaluate.py
"""Steps 5/6/9: run an ASR model on a test manifest and score it.

Examples:
    python src/evaluate.py --model whisper-small --limit 20      # quick check
    python src/evaluate.py --model whisper-small
    python src/evaluate.py --model mms
    python src/evaluate.py --model <hf-id-or-folder> --kind whisper --tag whisper-small-ft --language yoruba
    python src/evaluate.py --model mms --split ood_test

Outputs:
    results/predictions/<tag>__<split>.csv   one row per clip (ref, pred, per-clip WER/CER)
    results/metrics.csv                      one row per (model, split); re-runs replace the row
"""
import argparse
import sys
import time
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import torch
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parent))
from normalize import normalize  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
SR = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PRESETS = {
    "whisper-small": dict(model="openai/whisper-small", kind="whisper"),
    "mms": dict(model="facebook/mms-1b-all", kind="mms"),
    "akan-whisper": dict(model="GiftMark/akan-whisper-model", kind="whisper"),
}


class WhisperASR:
    def __init__(self, model_id, language=None):
        from transformers import WhisperForConditionalGeneration, WhisperProcessor
        self.processor = WhisperProcessor.from_pretrained(model_id)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_id).to(DEVICE).eval()
        # Twi is not a Whisper language: zero-shot uses auto-detection unless
        # a language is given; fine-tuned models use the proxy language they were trained with.
        self.gen = dict(task="transcribe", max_new_tokens=128)
        if language:
            self.gen["language"] = language

    @torch.inference_mode()
    def __call__(self, batch):
        feats = self.processor(batch, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE)
        ids = self.model.generate(feats, **self.gen)
        return self.processor.batch_decode(ids, skip_special_tokens=True)


class MmsASR:
    def __init__(self, model_id, lang="aka"):
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        self.processor = AutoProcessor.from_pretrained(model_id, target_lang=lang)
        self.model = Wav2Vec2ForCTC.from_pretrained(
            model_id, target_lang=lang, ignore_mismatched_sizes=True).to(DEVICE).eval()

    @torch.inference_mode()
    def __call__(self, batch):
        inputs = self.processor(batch, sampling_rate=SR, return_tensors="pt", padding=True).to(DEVICE)
        logits = self.model(**inputs).logits
        return self.processor.batch_decode(logits.argmax(dim=-1).cpu())


def score(refs, hyps, fold=False):
    r = [normalize(x, fold) for x in refs]
    h = [normalize(x, fold) for x in hyps]
    return jiwer.wer(r, h), jiwer.cer(r, h)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="preset name, HF id, or local folder")
    ap.add_argument("--kind", choices=["whisper", "mms"], help="needed when --model is not a preset")
    ap.add_argument("--tag", help="name used in results (default: preset name / folder name)")
    ap.add_argument("--split", default="test", help="manifest in data/processed/<split>.csv")
    ap.add_argument("--language", help="Whisper language token (fine-tuned models)")
    ap.add_argument("--mms-lang", default="aka")
    ap.add_argument("--limit", type=int, help="evaluate only the first N clips (smoke test)")
    ap.add_argument("--batch-size", type=int, default=8)
    args = ap.parse_args()

    preset = PRESETS.get(args.model, {})
    model_id = preset.get("model", args.model)
    kind = args.kind or preset.get("kind")
    if not kind:
        ap.error("--kind is required for a non-preset model")
    tag = args.tag or (args.model if preset else Path(args.model).name)

    df = pd.read_csv(ROOT / "data" / "processed" / f"{args.split}.csv")
    if args.limit:
        df = df.head(args.limit)
    print(f"{tag}: {len(df)} clips from {args.split} on {DEVICE}")

    asr = WhisperASR(model_id, args.language) if kind == "whisper" else MmsASR(model_id, args.mms_lang)

    preds, t0 = [], time.time()
    for i in tqdm(range(0, len(df), args.batch_size)):
        paths = df["audio"].iloc[i:i + args.batch_size]
        audio = [librosa.load(ROOT / p, sr=SR)[0] for p in paths]
        preds.extend(asr(audio))
    elapsed = time.time() - t0

    df["pred"] = [p.strip() for p in preds]
    df["ref_norm"] = df["text"].map(normalize)
    df["pred_norm"] = df["pred"].map(normalize)
    df["wer"] = [jiwer.wer(r, h) if r else None for r, h in zip(df["ref_norm"], df["pred_norm"])]
    df["cer"] = [jiwer.cer(r, h) if r else None for r, h in zip(df["ref_norm"], df["pred_norm"])]

    out = ROOT / "results" / "predictions"
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / f"{tag}__{args.split}.csv", index=False, encoding="utf-8")

    wer, cer = score(df["text"], df["pred"])
    wer_f, cer_f = score(df["text"], df["pred"], fold=True)
    row = {"model": tag, "model_id": model_id, "split": args.split, "clips": len(df),
           "wer": wer, "cer": cer, "wer_folded": wer_f, "cer_folded": cer_f}
    for col in ["dialect", "gender"]:
        if col in df:
            for val, g in df.groupby(col):
                row[f"wer_{val}"], row[f"cer_{val}"] = score(g["text"], g["pred"])
    audio_secs = df["duration"].sum() if "duration" in df else None
    row["real_time_factor"] = elapsed / audio_secs if audio_secs else None
    row = {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()}

    metrics = ROOT / "results" / "metrics.csv"
    old = pd.read_csv(metrics) if metrics.exists() else pd.DataFrame()
    if not old.empty and not args.limit:
        old = old[~((old["model"] == tag) & (old["split"] == args.split))]
    if not args.limit:  # smoke tests are not recorded
        pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(metrics, index=False)

    print(pd.Series(row).to_string())
    print("\nExamples:")
    for _, r in df.head(5).iterrows():
        print(f"  REF:  {r['text']}\n  PRED: {r['pred']}\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/finetune_whisper.py
"""Step 7: fine-tune Whisper-small on the TwiBench train split. Needs a GPU
(run it on Kaggle/Colab via notebooks/twibench_kaggle.ipynb).

The train/dev split is computed from the file names with the same hash as
prepare_data.py, so it always matches the local test set.

Usage (GPU, full train split):
    python src/finetune_whisper.py --output models/whisper-small-twi
Usage (CPU, local train sample, decoder only):
    python src/finetune_whisper.py --data local --freeze-encoder --batch-size 8 --grad-accum 1 \
        --max-steps 2000 --dev-clips 150 --eval-steps 500 --lr 3e-5
"""
import argparse
import io
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import jiwer
import librosa
import pandas as pd
import soundfile as sf
import torch
from datasets import Audio, Dataset, concatenate_datasets, load_dataset
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, WhisperProcessor)

sys.path.insert(0, str(Path(__file__).resolve().parent))
from normalize import normalize  # noqa: E402
from prepare_data import NAME_RE, SOURCES, bucket  # noqa: E402

ROOT = Path(__file__).resolve().parents[1]
SR = 16_000


def split_of(file_name: str, dialect: str) -> str:
    m = NAME_RE.match(file_name)
    if not m:
        return "unused"
    s, p = bucket(f"{dialect}:{m['speaker']}"), bucket(m["prompt"])
    return s if s == p else "unused"


def load_splits(dev_clips: int):
    parts = []
    for dialect, repo in SOURCES.items():
        ds = load_dataset(repo)
        ds = concatenate_datasets([ds[s] for s in ds])          # we re-split ourselves
        ds = ds.cast_column("audio_path", Audio(decode=False))  # decoded in the collator
        ds = ds.map(lambda b: {"split": [split_of(a["path"], dialect) for a in b["audio_path"]]},
                    batched=True, num_proc=4)
        parts.append(ds)
    ds = concatenate_datasets(parts)
    train = ds.filter(lambda s: s == "train", input_columns="split", num_proc=4)
    dev = ds.filter(lambda s: s == "dev", input_columns="split", num_proc=4)
    dev = dev.shuffle(seed=0).select(range(min(dev_clips, len(dev))))
    print(f"train {len(train)} clips | dev subset {len(dev)} clips")
    return train, dev


def load_local_splits(dev_clips: int):
    """CPU mode: the train sample and dev clips exported by prepare_data.py."""
    def read(split):
        df = pd.read_csv(ROOT / "data" / "processed" / f"{split}.csv")
        return Dataset.from_pandas(df[["audio", "text"]].rename(columns={"text": "transcription"}),
                                   preserve_index=False)
    train, dev = read("train"), read("dev")
    dev = dev.shuffle(seed=0).select(range(min(dev_clips, len(dev))))
    print(f"train {len(train)} clips | dev subset {len(dev)} clips (local)")
    return train, dev


def load_audio(row):
    if "audio_path" not in row:  # local FLAC, already 16 kHz mono
        return sf.read(ROOT / row["audio"], dtype="float32")[0]
    y, sr = sf.read(io.BytesIO(row["audio_path"]["bytes"]), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    return y if sr == SR else librosa.resample(y, orig_sr=sr, target_sr=SR)


@dataclass
class Collator:
    """Decodes audio and builds log-mel features on the fly (no feature cache on disk)."""
    processor: WhisperProcessor

    def __call__(self, rows):
        audio = [load_audio(r) for r in rows]
        batch = self.processor.feature_extractor(audio, sampling_rate=SR, return_tensors="pt")
        labels = self.processor.tokenizer([r["transcription"].strip() for r in rows],
                                          padding=True, return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        sot = self.processor.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
        if (ids[:, 0] == sot).all():  # the model adds it back as decoder start token
            ids = ids[:, 1:]
        batch["labels"] = ids
        return batch


class TimeBudget(TrainerCallback):
    """Stop training cleanly once a wall-clock budget is used up."""

    def __init__(self, hours: float):
        self.deadline = time.time() + hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print(f"Time budget reached at step {state.global_step}; stopping.", flush=True)
            control.should_training_stop = True
        return control


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base-model", default="openai/whisper-small")
    ap.add_argument("--output", default="models/whisper-small-twi")
    ap.add_argument("--language", default="yoruba",
                    help="proxy language token (Twi is not a Whisper language); use the same one at evaluation")
    ap.add_argument("--max-steps", type=int, default=4000)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--grad-accum", type=int, default=2)
    ap.add_argument("--lr", type=float, default=1e-5)
    ap.add_argument("--dev-clips", type=int, default=600)
    ap.add_argument("--eval-steps", type=int, default=500)
    ap.add_argument("--data", choices=["hf", "local"], default="hf",
                    help="hf: full train split streamed from Hugging Face (GPU); "
                         "local: the train sample exported by prepare_data.py (CPU)")
    ap.add_argument("--freeze-encoder", action="store_true",
                    help="train the decoder only (much faster; used for CPU runs)")
    ap.add_argument("--warmup-steps", type=int, help="default: min(400, max_steps / 10)")
    ap.add_argument("--max-hours", type=float,
                    help="stop training after this many hours (uses a constant LR so stopping early is clean)")
    args = ap.parse_args()
    gpu = torch.cuda.is_available()

    processor = WhisperProcessor.from_pretrained(args.base_model, language=args.language, task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(args.base_model)
    model.generation_config.language = args.language
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
    model.config.use_cache = False
    if args.freeze_encoder:
        model.freeze_encoder()
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable parameters: {n_train / 1e6:.0f}M | device: {'cuda' if gpu else 'cpu'}")

    train, dev = load_splits(args.dev_clips) if args.data == "hf" else load_local_splits(args.dev_clips)

    def compute_metrics(pred):
        ids = pred.label_ids.copy()
        ids[ids == -100] = processor.tokenizer.pad_token_id
        hyp = [normalize(t) for t in processor.batch_decode(pred.predictions, skip_special_tokens=True)]
        ref = [normalize(t) for t in processor.batch_decode(ids, skip_special_tokens=True)]
        return {"wer": jiwer.wer(ref, hyp), "cer": jiwer.cer(ref, hyp)}

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        per_device_eval_batch_size=16,
        learning_rate=args.lr,
        warmup_steps=args.warmup_steps if args.warmup_steps is not None else min(400, args.max_steps // 10),
        lr_scheduler_type="constant_with_warmup" if args.max_hours else "linear",
        max_steps=args.max_steps,
        fp16=gpu,
        gradient_checkpointing=gpu and not args.freeze_encoder,
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        # On a small local disk, skip ~1-2 GB checkpoints and keep only the final model.
        save_strategy="steps" if gpu else "no",
        save_steps=args.eval_steps,
        save_total_limit=2,
        logging_steps=50 if gpu else 10,
        predict_with_generate=True,
        generation_max_length=128,
        load_best_model_at_end=gpu,
        metric_for_best_model="wer",
        greater_is_better=False,
        report_to="none",
        dataloader_num_workers=4 if gpu else 0,
        remove_unused_columns=False,  # the collator needs the raw audio + text columns
        seed=42,
    )
    trainer = Seq2SeqTrainer(
        model=model, args=training_args,
        train_dataset=train, eval_dataset=dev,
        data_collator=Collator(processor),
        compute_metrics=compute_metrics,
        processing_class=processor,
        callbacks=[TimeBudget(args.max_hours)] if args.max_hours else None,
    )
    trainer.train()

    model.config.use_cache = True
    trainer.save_model(args.output)
    processor.save_pretrained(args.output)
    pd.DataFrame(trainer.state.log_history).to_csv(Path(args.output) / "training_log.csv", index=False)
    print("Best dev result:", trainer.evaluate())


if __name__ == "__main__":
    main()


## 1. Install

In [ ]:
!pip install -q "transformers==5.20.1" "accelerate>=0.34" "jiwer>=3.0" soundfile librosa

## 2. Data: download, speaker- and sentence-disjoint split, out-of-domain set

In [ ]:
!python src/prepare_data.py
!python src/prepare_ood.py

## 3. Zero-shot baselines

In [ ]:
if RUN_BASELINES:
    for model in ["whisper-small", "mms"]:
        for split in ["test", "ood_test"]:
            !python src/evaluate.py --model {model} --split {split} --batch-size 16

## 4. Fine-tune Whisper-small and evaluate it

In [ ]:
if RUN_FINETUNE:
    !python src/finetune_whisper.py --output models/whisper-small-twi --max-steps {MAX_STEPS}
    for split in ["test", "ood_test"]:
        !python src/evaluate.py --model models/whisper-small-twi --kind whisper --tag whisper-small-ft --language yoruba --split {split} --batch-size 16

## 5. Package results for download

In [ ]:
import shutil, glob, pandas as pd
shutil.copy("data/processed/metadata.csv", "results/metadata.csv")
for f in glob.glob("models/whisper-small-twi/training_log.csv"):
    shutil.copy(f, "results/training_log.csv")
shutil.make_archive(f"{OUT_DIR}/twibench_results", "zip", "results")
if RUN_FINETUNE and SAVE_MODEL_ZIP:
    for d in glob.glob("models/whisper-small-twi/checkpoint-*"):
        shutil.rmtree(d)
    shutil.make_archive(f"{OUT_DIR}/whisper-small-twi", "zip", "models/whisper-small-twi")
if not IN_COLAB:  # on Kaggle, keep Output small
    shutil.rmtree("data", ignore_errors=True)
    shutil.rmtree("models", ignore_errors=True)
print("Saved to", OUT_DIR, os.listdir(OUT_DIR))
pd.read_csv("results/metrics.csv")